<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Main-Version-2/mnps_job_equity_prompt_update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNPS Job Equity Prompt Update
> A notebook that builds on the work done in the Hackathon
> DSI DSSG + MNPS   
> September 4, 2025  
> Drafted by Wayne Birch - [contact him](wayne.birch@mnps.org) for questions, code update needs, or other questions about the notebook!

This notebook used the starting point from the mini Hackathon with Metro Nashville Public Schools (MNPS) and the VU Data Science Institute (VU DSI). Updates have been made to file locations and refinements made to improve the model's results.

## **1** | Notebook Parameters
* **Outcome and evaluation**: Runs are validated by the MNPS project team and downstream automation (no external judges).
* A successful run must:
  * Read inputs exactly from /content/Ground Truth Masterfile.csv and /content/Sample File.csv.
  * Use the Ground Truth Masterfile as few-shot exemplars (TF-IDF top-k retrieval) to guide classification of every row in Sample File.csv.
  * Write a canonical output /content/run_artifacts/predictions.csv and a convenience copy /content/predictions.csv.
  * Copy all run artifacts to /content/drive/My Drive/Colab Notebooks/Run Results/RUN_<timestamp>.
* Required columns in predictions.csv: run_id, job_title_original, new_job_title, major_role_group, minor_sub_group, grouping_justification
(Additional columns are allowed.)
*Optional: If an adjudication sheet MNPS_Adjudication_Sheet_<RUN_ID>.csv is present, the notebook may also emit merged/scored outputs (e.g., predictions_scored.csv, errors_only.csv) for internal QA.

* **Objective**:
* Build a reproducible, reliable, automatable system that classifies MNPS job descriptions into:
  * New Job Title (e.g., “Collections Specialist II”)
  * Major Role Group (Specialist, Analyst, Director, Manager, Technician, Coordinator)
  * Minor Sub-Group (I, II, III, IV where appropriate)
* Grouping Justification (brief rationale citing Position Summary, Essential Functions, Education, Experience, Licenses/Certifications, KSAs)
* Classification rules:
  * Down-weight literal job-title strings appearing in summaries/EF; focus on what the job does (scope, decision latitude, supervision, consequences of error).
  * Up-weight licensure and scope of responsibility when present.
  * Bias toward verified MNPS outcomes using Ground Truth exemplars (top-k similar rows).
  * Run in batch over Sample File.csv with conservative settings (temperature=0.2) for reproducibility.

* **Usability & Reproducibility**
  * The notebook is Colab-ready with an Open in Colab badge at the top.
  * Expected manual steps: upload the two CSVs to /content/ and set OPENAI_API_KEY.
  * No code edits should be required to execute end-to-end.


## **2** | Environment Setup
Again, you're completely free to just download this notebook, create a local virtual environment and get to coding in your favorite IDE. We provide this code just as a rapid method to get started, and focus our efforts on implementation through Google Colab.

### **2a** | API Key Setup
#### **2a.1** | Access
The DSI has provided you an API key which can access **some** of the OpenAI models. These include:
* All versions of gpt-4o
* All versions of gpt-4.1
* All versions of o3-mini

Vector store upload, web search, code interpreter, and other functionality outside of the Chat Completions and Messages API is **not** supported. If you really want to use these things, you will have to make a good and cost-supported argument. If you don't feel like arguing, you can also utilize your own OpenAI API key.

#### **2a.2** | API Keys in Google Colab
To use your API key, click on the key icon (looks sort of like 🔑) in the left sidebar.  Under **Name**, add `OPENAI_API_KEY`. Under **Value**, paste your API key. Your API key is a jumble of numbers and letters, maybe even other symbols. Click the slider checkbox to enable **Notebook access** (so your notebook will grab these values without asking you).  

### **2b** | Runtime setup
We're going to install some packages in your environment so that you have access to the code functionality. If you need more packages, install more packages. Install **only** packages you trust.

In [1]:
!pip install openai

In [2]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List
import pandas as pd
from google.colab import userdata

# set OpenAI API key environment variable using Google Colab
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## **3** | The Data

The current prompt is a two-step prompt that is successful through the ChatGPT interface. It requires two types of data:
* The data to be classified
* Supporting resources

We need to read all of this in. Let's grab it and use it. The first thing you'll do is just straight up download a zip file of all of this information.

You can download all of the reference files from the link provided, then upload in the sidebar. You'll then unzip the directory using the code below.

Click on the folder icon in the left sidebar (kinda looks like this 🗂️) and you'll see all the files there. We'll read them in.


In [3]:
!unzip "/content/MNPS_Prompt_Resources.zip"

Archive:  /content/MNPS_Prompt_Resources.zip
replace Korn_Ferry Lominger 38 Competencies.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: Korn_Ferry Lominger 38 Competencies.csv  
  inflating: Competency Extended Descriptions.csv  
  inflating: MNPS KSACs.csv          
  inflating: MNPS Roles.csv          


In [4]:
resources_dir_prefix = '/content/'
roles_lookup = pd.read_csv(resources_dir_prefix+"MNPS Roles.csv")
determinants = pd.read_csv(resources_dir_prefix+"Competency Extended Descriptions.csv", encoding='latin1')
ksac_table = pd.read_csv(resources_dir_prefix+"MNPS KSACs.csv")
korn_ferry = pd.read_csv(resources_dir_prefix+"Korn_Ferry Lominger 38 Competencies.csv", encoding='latin1')

## **4** | The Prompts

What we have here is a direct prompt to get the response that we're looking for. We'll make this happen directly using the OpenAI Chat Completions API. Note that you can use other APIs as you like.

In [5]:
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "Job Description Export Specialists.xlsx" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-groupings (e.g., Specialist I, II, III, IV) - not to exceed level IV
- Use the MNPS Roles and MNPS KSACs documents to help you determine major role groupings.
- Use the remaining documents to help you clarify subtle differences in role groupings and sub-groupings.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III")

Additional Guidelines:

- Ensure all sources used are cited properly.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks.

"""

Instead of asking for a table output, we will use **structured outputs**. Though this is a common approach for the outputs of LLMs/AI systems, you can learn more about this on [OpenAI's structured output documentation](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses). Note that you can find this information on almost all LLM/AI platform or package providers.

In [6]:
from pydantic import BaseModel, Field

class JobClassification(BaseModel):
    """Represents the classification of a job based on its functions."""
    job_title_original: str = Field(..., description="The original job title as provided in the input data using the job title convention specified.")
    new_job_title: str = Field(..., description="The proposed new job title based on the classification using the job title convention specified.")
    major_role_group: str = Field(..., description="The major grouping of the job based on its functional role (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="The minor sub-grouping within the major role group (e.g., Specialist I, II, III, IV).")
    grouping_justification: str = Field(..., description="The justification for placing the job in the specific major and minor groups, referencing job attributes and relevant documents.")

In [7]:
class JobClassificationTable(BaseModel):
  """The table classification and overall commentary on the groupings provided by the AI system."""
  job_classification_table: List[JobClassification] = Field(..., description="The table of job classifications.")
  narrative_rationale: str = Field(..., description="The narrative commentary on the groupings provided by the AI system.")

Create classifications using OpenAI. Of note here is:
* The **developer** prompt - this is the "system prompt" or "custom instructions" for the model. This determines the overall behavior of the model.
* The **user** prompt - this is what we send to the model like when we're chatting with ChatGPT.

In [8]:
# Create openAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Create messages to send
messages = [
    {"role": "developer", "content": zero_shot_prompt},
    {"role": "user", "content": "Classify the following job description: [Paste Job Description Here]"} # Replace with actual job description
]

# Assuming JobClassification and zero_shot_prompt are defined in the preceding code
response = client.beta.chat.completions.parse(
    model="gpt-4o", # Or another available model
    messages=messages,
    temperature=1,
    max_tokens=1000,
    response_format=JobClassificationTable
)

print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-CCBlXA7JTSSWZtZmu9rEXXcBCfetm",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"job_classification_table\":[{\"job_title_original\":\"Import/Export Specialist\",\"new_job_title\":\"Logistics Specialist II\",\"major_role_group\":\"Specialist\",\"minor_sub_group\":\"II\",\"grouping_justification\":\"The job requires specialized knowledge in logistics coordination, which aligns with the 'Specialist' major role group. The tasks involve managing import and export processes, indicating a level of complexity and responsibility that fits the 'II' sub-group, evidenced by required work experience and essential functions emphasizing coordination and compliance.\"}],\"narrative_rationale\":\"The provided Import/Export Specialist role was evaluated on its key responsibilities such as managing logistics, handling documentation, and ensuring compliance with regulations. These tasks align it wit

In [9]:
#look at response
response.choices[0].message.parsed

JobClassificationTable(job_classification_table=[JobClassification(job_title_original='Import/Export Specialist', new_job_title='Logistics Specialist II', major_role_group='Specialist', minor_sub_group='II', grouping_justification="The job requires specialized knowledge in logistics coordination, which aligns with the 'Specialist' major role group. The tasks involve managing import and export processes, indicating a level of complexity and responsibility that fits the 'II' sub-group, evidenced by required work experience and essential functions emphasizing coordination and compliance.")], narrative_rationale="The provided Import/Export Specialist role was evaluated on its key responsibilities such as managing logistics, handling documentation, and ensuring compliance with regulations. These tasks align it with the 'Specialist' role group, reflecting a focus on functional expertise. Given the level of work experience required and the responsibilities involving international coordination

We can make this into a table using pandas!

In [10]:
response_list = response.choices[0].message.parsed.job_classification_table
response_dict_list = [item.model_dump() for item in response_list]

In [11]:
# see outputs
pd.DataFrame(response_dict_list)

,job_title_original,new_job_title,major_role_group,minor_sub_group,grouping_justification
0,Import/Export Specialist,Logistics Specialist II,Specialist,II,The job requires specialized knowledge in logi...


# Task
Load job descriptions from "New Sample_08.07.2025.csv" and process them in batches using the OpenAI API.

## Load job descriptions

### Subtask:
Read the job descriptions from the "New Sample_08.07.2025.csv" file into a pandas DataFrame.


**Reasoning**:
Read the job descriptions from the specified CSV file into a pandas DataFrame and display the head and columns to confirm successful loading.



In [17]:
job_descriptions_df = pd.read_csv('/content/New Sample_08.07.2025.csv', encoding='latin1')
display(job_descriptions_df.head())
display(job_descriptions_df.columns)

,Job Description Name,Position Summary,Education,Work Experience,Essential Functions,Licenses and Certifications,"Knowledge, Skills and Abilities"
0,Tech Mail Center,Sorts and delivers incoming and outgoing mail ...,High School Diploma or GED Required,NaN,Pickup from and delivery to USPS locations inc...,"DL NUMBER - Driver License, Valid and in State...",NaN
1,Coord Safe and Drug Free,Perform a broad range of duties that promotes ...,Master's Degree from an accredited institution...,Experience managing a long-term project in an...,Management of the 1st Time Drug Offender and...,NaN,Strong interpersonal and communication skills....
2,Application Systems Analyst- Kronos,Serves as the subject matter expert and main c...,NaN,4-6 years Experience in a Kronos support role ...,Initiates projects by gathering and analyzing ...,NaN,Knowledge and understanding of Kronos workflow...
3,Assistant Financial Analyst,Supports the financial analysis and planning a...,NaN,less than 1 year Experience working in a finan...,Collects financial data from a variety of sour...,NaN,Knowledge and understanding of basic accountin...
4,Analyst Payroll Compliance,This position ensures payroll accuracy and com...,"Bachelor's Degree Accounting, Finance, Human R...","4-6 years Experience in payroll processing, au...",Serve as a subject matter expert and resource ...,Certified Payroll Professional (CPP)-APA with...,Ability to identify trends in payroll errors a...


Index(['Job Description Name', 'Position Summary', 'Education',
       'Work Experience', 'Essential Functions', 'Licenses and Certifications',
       'Knowledge, Skills and Abilities'],
      dtype='object')

**Reasoning**:
The previous attempt to read the CSV failed due to a UnicodeDecodeError. I will try reading the CSV again, specifying a different encoding that might handle the characters in the file. Given the error message, 'latin1' is a common alternative that often resolves such issues.



In [14]:
job_descriptions_df = pd.read_csv('/content/New Sample_08.07.2025.csv', encoding='latin1')
display(job_descriptions_df.head())
display(job_descriptions_df.columns)

,Job Description Name,Position Summary,Education,Work Experience,Essential Functions,Licenses and Certifications,"Knowledge, Skills and Abilities"
0,Tech Mail Center,Sorts and delivers incoming and outgoing mail ...,High School Diploma or GED Required,NaN,Pickup from and delivery to USPS locations inc...,"DL NUMBER - Driver License, Valid and in State...",NaN
1,Coord Safe and Drug Free,Perform a broad range of duties that promotes ...,Master's Degree from an accredited institution...,Experience managing a long-term project in an...,Management of the 1st Time Drug Offender and...,NaN,Strong interpersonal and communication skills....
2,Application Systems Analyst- Kronos,Serves as the subject matter expert and main c...,NaN,4-6 years Experience in a Kronos support role ...,Initiates projects by gathering and analyzing ...,NaN,Knowledge and understanding of Kronos workflow...
3,Assistant Financial Analyst,Supports the financial analysis and planning a...,NaN,less than 1 year Experience working in a finan...,Collects financial data from a variety of sour...,NaN,Knowledge and understanding of basic accountin...
4,Analyst Payroll Compliance,This position ensures payroll accuracy and com...,"Bachelor's Degree Accounting, Finance, Human R...","4-6 years Experience in payroll processing, au...",Serve as a subject matter expert and resource ...,Certified Payroll Professional (CPP)-APA with...,Ability to identify trends in payroll errors a...


Index(['Job Description Name', 'Position Summary', 'Education',
       'Work Experience', 'Essential Functions', 'Licenses and Certifications',
       'Knowledge, Skills and Abilities'],
      dtype='object')

## Prepare data for api

### Subtask:
Iterate through the DataFrame and format the job descriptions into the structure required for the OpenAI API calls.


**Reasoning**:
Iterate through the DataFrame and format the job descriptions into the required structure.



In [15]:
job_description_list = []
for index, row in job_descriptions_df.iterrows():
    job_description = ""
    for col in ['Position Summary', 'Education', 'Work Experience', 'Essential Functions', 'Licenses and Certifications', 'Knowledge, Skills and Abilities']:
        if pd.notna(row[col]):
            job_description += str(row[col]) + " "
    job_description = job_description.strip()
    job_description_list.append({
        'original_job_title': row['Job Description Name'],
        'job_description': job_description
    })

# Display the first few formatted job descriptions to verify
print(job_description_list[:5])

[{'original_job_title': 'Tech Mail Center', 'job_description': 'Sorts and delivers incoming and outgoing mail and other materials requiring distribution. Operates Mail Center equipment including postage meter and mail folder/inserter.\r\n High School Diploma or GED Required Pickup from and delivery to USPS locations including bulk, permits, etc.\nAccountable for special classes of mail such as certified, overnight letters, FedEx, and UPS.\nTrains exceptional education students through the MNPS Transition Program to perform duties in a mail center environment.\nPerforms pre-delivery sorting and inspection and schedules routing for delivery of mail and packages.\nVerifies and corrects misdirected mail.\nLoads and unloads Mail Center fleet.\nAdvises staff on Mail Center policies and procedures. DL NUMBER - Driver License, Valid and in State Valid Tennessee Drivers\x92 License  Required\n Basic Mail Certification   Required'}, {'original_job_title': 'Coord Safe and Drug Free', 'job_descrip

## Process in batches

### Subtask:
Send the job descriptions to the OpenAI API in batches to avoid hitting API limits and manage processing time.


**Reasoning**:
Iterate through the job descriptions in batches, construct the API request messages for each batch, and send the requests to the OpenAI API, handling potential errors and delays.



In [ ]:
import time

batch_size = 10  # Adjust batch size as needed
classified_jobs = []
errors = []

for i in range(0, len(job_description_list), batch_size):
    batch = job_description_list[i:i + batch_size]
    messages = [
        {"role": "system", "content": zero_shot_prompt}
    ]
    # Include original job title in the message for better tracking
    for job in batch:
        messages.append({"role": "user", "content": f"Classify the following job description (Original Title: {job['original_job_title']}): {job['job_description']}"})


    try:
        response = client.beta.chat.completions.parse(
            model="gpt-4o",
            messages=messages,
            temperature=0.5, # Lower temperature for more consistent results
            max_tokens=1500, # Increase max tokens to accommodate multiple classifications
            response_format=JobClassificationTable
        )
        # Assuming the response contains a list of classifications
        if response.choices and response.choices[0].message.parsed:
             for item in response.choices[0].message.parsed.job_classification_table:
                 # Extract the original job title from the grouping justification
                 original_title = "Unknown Original Title"
                 # A more robust way would be to have the API return the original title directly,
                 # but given the current Pydantic model, we'll try to extract it from the justification
                 # This approach is still not ideal and should be improved if the API model can be changed.
                 # For now, let's iterate through the original batch to find the matching description
                 # and use its original title.
                 matched_job = next((job for job in batch if job['job_description'] in item.grouping_justification), None)
                 if matched_job:
                     original_title = matched_job['original_job_title']
                 else:
                     # As a fallback, try to find a match based on the new job title or major role group
                     matched_job = next((job for job in batch if item.new_job_title in job['original_job_title'] or item.major_role_group in job['original_job_title']), None)
                     if matched_job:
                         original_title = matched_job['original_job_title']


                 classified_jobs.append({
                     'original_job_title': original_title,
                     'new_job_title': item.new_job_title,
                     'major_role_group': item.major_role_group,
                     'minor_sub_group': item.minor_sub_group,
                     'grouping_justification': item.grouping_justification
                 })

        else:
            errors.append(f"No parsed response for batch starting at index {i}")


    except Exception as e:
        errors.append(f"Error processing batch starting at index {i}: {e}")
        # Implement a delay before the next request in case of rate limits or errors
        time.sleep(60) # Wait for 60 seconds before the next attempt

    # Add a delay between batches to avoid hitting rate limits
    time.sleep(10) # Wait for 10 seconds between batches

# Optionally, convert classified_jobs to a DataFrame
classified_jobs_df = pd.DataFrame(classified_jobs)

# Display the first few classified jobs and any errors encountered
print("Classified Jobs (first 5):")
display(classified_jobs_df.head())
print("\nErrors Encountered:")
print(errors)

## Review and analyze results

### Subtask:
Examine the generated classifications and narrative rationale.


**Reasoning**:
I need to examine the generated classifications and the narrative rationale as per the instructions. This involves looking at the distribution of the classifications in the dataframe, sampling the justifications, and printing the overall narrative.



In [ ]:
# 1. Review the classified_jobs_df DataFrame. Look at the distribution of major role groups and minor sub-groups.
print("Major Role Group Distribution:")
display(classified_jobs_df['major_role_group'].value_counts())

print("\nMinor Sub-Group Distribution:")
display(classified_jobs_df['minor_sub_group'].value_counts())

# 2. Examine the grouping_justification column for a sample of entries.
print("\nSample Grouping Justifications:")
display(classified_jobs_df[['original_job_title', 'new_job_title', 'major_role_group', 'minor_sub_group', 'grouping_justification']].sample(5))

# 3. Print or display the narrative_rationale from the last successful API response.
# Assuming 'response' from the previous cell holds the last successful response
print("\nNarrative Rationale:")
# Access the narrative rationale from the parsed response
narrative_rationale = response.choices[0].message.parsed.narrative_rationale
print(narrative_rationale)

## Summary:

### Data Analysis Key Findings

*   The dataset was successfully loaded from "New Sample\_08.07.2025.csv" using the 'latin1' encoding.
*   Job descriptions were formatted by concatenating relevant columns ('Position Summary', 'Education', 'Work Experience', 'Essential Functions', 'Licenses and Certifications', 'Knowledge, Skills and Abilities') for each record, handling missing values.
*   Job descriptions were successfully sent to the OpenAI API in batches of 10 using the "gpt-4o" model and the `JobClassificationTable` response format.
*   The API responses were parsed, and classification information (new job title, major role group, minor sub group, grouping justification) was extracted and stored in a DataFrame.
*   Common major role groups identified by the API included 'Specialist', 'Analyst', and 'Manager'.
*   The `grouping_justification` column provided brief explanations for the classifications, often referencing job duties and experience.
*   A narrative rationale explaining the classification process for a specific job was successfully retrieved from the API response.

### Insights or Next Steps

*   Review the `grouping_justification` and `narrative_rationale` more extensively to assess the quality and consistency of the API's reasoning.
*   Implement a more robust method to match the API's classified results back to the original job titles, as the current method (`job['job_description'] in item.grouping_justification or item.new_job_title in job['original_job_title'] or item.major_role_group in job['original_job_title']`) might not always be accurate.


# Task
Load job descriptions from "New Sample_08.07.2025.csv", process them in batches using the OpenAI API to classify each job, and save the results to a CSV file in the same format as the "Sample Grouping Justifications" table, ensuring all 114 records are included in the output.

## Load job descriptions

### Subtask:
Read the job descriptions from the "New Sample_08.07.2025.csv" file into a pandas DataFrame.


## Prepare data for api

### Subtask:
Iterate through the DataFrame and format the job descriptions into the structure required for the OpenAI API calls.


**Reasoning**:
Iterate through the DataFrame and format the job descriptions into the required structure.



In [ ]:
job_description_list = []
for index, row in job_descriptions_df.iterrows():
    job_description = ""
    for col in ['Position Summary', 'Education', 'Work Experience', 'Essential Functions', 'Licenses and Certifications', 'Knowledge, Skills and Abilities']:
        if pd.notna(row[col]):
            job_description += str(row[col]) + " "
    job_description = job_description.strip()
    job_description_list.append({
        'original_job_title': row['Job Description Name'],
        'job_description': job_description
    })

# Display the first few formatted job descriptions to verify
print(job_description_list[:5])

## Collect and structure results

### Subtask:
Collect the responses from the API calls and structure them into a pandas DataFrame containing all classified jobs.


**Reasoning**:
I need to collect the API responses and structure them into a pandas DataFrame as requested by the subtask. The `classified_jobs` list already contains the dictionaries with the classified job data from the previous API calls. I just need to convert this list into a DataFrame and display the head and the number of rows to complete the subtask.



In [ ]:
# The classified_jobs list already contains the dictionaries of classified job data from the previous step.
# Convert the list of dictionaries into a pandas DataFrame.
classified_jobs_df = pd.DataFrame(classified_jobs)

# Display the head of the resulting DataFrame to verify its structure and content.
print("Head of the DataFrame with all classified jobs:")
display(classified_jobs_df.head())

# Print the number of rows in the DataFrame to confirm all classified jobs are included.
print(f"\nTotal number of classified jobs: {len(classified_jobs_df)}")

## Save results to csv

### Subtask:
Save the final DataFrame containing all classified jobs to a CSV file.


**Reasoning**:
Save the classified jobs DataFrame to a CSV file without the index.



In [ ]:
classified_jobs_df.to_csv('classified_job_descriptions.csv', index=False)

## Review and analyze results

### Subtask:
Examine the generated classifications in the DataFrame.


**Reasoning**:
Display the first 5 rows of the classified_jobs_df DataFrame, print the number of rows to confirm all 114 original job descriptions have a corresponding classification, and display the value counts for the 'major_role_group' and 'minor_sub_group' columns.



In [ ]:
display(classified_jobs_df.head())
print(f"Number of classified jobs: {len(classified_jobs_df)}")
print("\nMajor Role Group Distribution:")
display(classified_jobs_df['major_role_group'].value_counts())
print("\nMinor Sub-Group Distribution:")
display(classified_jobs_df['minor_sub_group'].value_counts())

**Reasoning**:
Display a random sample of 10 rows from the classified_jobs_df DataFrame to qualitatively assess the classification results and justifications.



In [ ]:
display(classified_jobs_df[['original_job_title', 'new_job_title', 'major_role_group', 'minor_sub_group', 'grouping_justification']].sample(10))

## Summary:

### Data Analysis Key Findings

*   The initial dataset "New Sample\_08.07.2025.csv" containing 114 job descriptions was successfully loaded.
*   Job descriptions were formatted by concatenating relevant columns for processing.
*   The job descriptions were processed in batches using the OpenAI API, and the results were collected.
*   The classified results were structured into a pandas DataFrame, which contained columns for original and new job titles, major and minor role groups, and grouping justifications.
*   The resulting DataFrame contained 92 classified jobs, which is less than the original 114 records, indicating that some job descriptions were not classified or included in the output.
*   The major role groups identified included 'Specialist', 'Analyst', and 'Manager' among others.
*   The minor sub-groups included various levels (I, II, III) and specific sub-groups.
*   The classified results, including new job titles, major/minor groups, and justifications, were saved to a CSV file named "classified\_job\_descriptions.csv".

### Insights or Next Steps

*   Investigate why only 92 out of the 114 original records were classified and included in the final output.
*   Review the classifications for accuracy and potentially refine the prompt or process for the OpenAI API calls to improve results or handle edge cases.


## Collect and structure results

### Subtask:
Collect the responses from the API calls and structure them into a pandas DataFrame containing all classified jobs.

**Reasoning**:
I need to collect the API responses and structure them into a pandas DataFrame as requested by the subtask. The `classified_jobs` list already contains the dictionaries with the classified job data from the previous API calls. I just need to convert this list into a DataFrame and display the head and the number of rows to complete the subtask.

In [ ]:
# The classified_jobs list already contains the dictionaries of classified job data from the previous step.
# Convert the list of dictionaries into a pandas DataFrame.
classified_jobs_df = pd.DataFrame(classified_jobs)

# Display the head of the resulting DataFrame to verify its structure and content.
print("Head of the DataFrame with all classified jobs:")
display(classified_jobs_df.head())

# Print the number of rows in the DataFrame to confirm all classified jobs are included.
print(f"\nTotal number of classified jobs: {len(classified_jobs_df)}")

## Save results to csv

### Subtask:
Save the final DataFrame containing all classified jobs to a CSV file.

**Reasoning**:
Save the classified jobs DataFrame to a CSV file without the index.

In [ ]:
classified_jobs_df.to_csv('classified_job_descriptions.csv', index=False)

## Review and analyze results

### Subtask:
Examine the generated classifications in the DataFrame.

**Reasoning**:
Display the first 5 rows of the classified_jobs_df DataFrame, print the number of rows to confirm all 114 original job descriptions have a corresponding classification, and display the value counts for the 'major_role_group' and 'minor_sub_group' columns.

In [ ]:
display(classified_jobs_df.head())
print(f"Number of classified jobs: {len(classified_jobs_df)}")
print("\nMajor Role Group Distribution:")
display(classified_jobs_df['major_role_group'].value_counts())
print("\nMinor Sub-Group Distribution:")
display(classified_jobs_df['minor_sub_group'].value_counts())

**Reasoning**:
Display a random sample of 10 rows from the classified_jobs_df DataFrame to qualitatively assess the classification results and justifications.

In [ ]:
display(classified_jobs_df[['original_job_title', 'new_job_title', 'major_role_group', 'minor_sub_group', 'grouping_justification']].sample(10))

## Summary:

### Data Analysis Key Findings

* The initial dataset "New Sample\_08.07.2025.csv" containing 114 job descriptions was successfully loaded.
* Job descriptions were formatted by concatenating relevant columns for processing.
* The job descriptions were processed in batches using the OpenAI API, and the results were collected.
* The classified results were structured into a pandas DataFrame, which contained columns for original and new job titles, major and minor role groups, and grouping justifications.
* The resulting DataFrame contained {num\_classified\_jobs} classified jobs, which is {comparison\_to\_114} the original 114 records{discrepancy\_note}.
* The major role groups identified included 'Specialist', 'Analyst', and 'Manager' among others.
* The minor sub-groups included various levels (I, II, III) and specific sub-groups.
* The classified results, including new job titles, major/minor groups, and justifications, were saved to a CSV file named "classified\_job\_descriptions.csv".

### Insights or Next Steps

* Investigate why {discrepancy\_reasoning} if all 114 records were not classified and included in the final output.
* Review the classifications for accuracy and potentially refine the prompt or process for the OpenAI API calls to improve results or handle edge cases.

## Load job descriptions

### Subtask:
Read the job descriptions from the "New Sample_08.07.2025.csv" file into a pandas DataFrame.

**Reasoning**:
Read the job descriptions from the specified CSV file into a pandas DataFrame and display the head and columns to confirm successful loading.

In [ ]:
job_descriptions_df = pd.read_csv('/content/New Sample_08.07.2025.csv', encoding='latin1')
display(job_descriptions_df.head())
display(job_descriptions_df.columns)

## Prepare data for api

### Subtask:
Iterate through the DataFrame and format the job descriptions into the structure required for the OpenAI API calls.

**Reasoning**:
Iterate through the DataFrame and format the job descriptions into the required structure.

In [ ]:
job_description_list = []
for index, row in job_descriptions_df.iterrows():
    job_description = ""
    for col in ['Position Summary', 'Education', 'Work Experience', 'Essential Functions', 'Licenses and Certifications', 'Knowledge, Skills and Abilities']:
        if pd.notna(row[col]):
            job_description += str(row[col]) + " "
    job_description = job_description.strip()
    job_description_list.append({
        'original_job_title': row['Job Description Name'],
        'job_description': job_description
    })

# Display the first few formatted job descriptions to verify
print(job_description_list[:5])

## Process in Batches

### Subtask:
Send the job descriptions to the OpenAI API in batches to avoid hitting API limits and manage processing time, ensuring each classified job is correctly associated with its original job title.

**Reasoning**:
Iterate through the job descriptions in batches, construct the API request messages for each batch, and send the requests to the OpenAI API, handling potential errors and delays, and ensuring original job titles are correctly associated.

In [ ]:
import time

batch_size = 10  # Adjust batch size as needed
classified_jobs = []
errors = []

for i in range(0, len(job_description_list), batch_size):
    batch = job_description_list[i:i + batch_size]
    messages = [
        {"role": "system", "content": zero_shot_prompt}
    ]
    # Include original job title in the message for better tracking
    for job in batch:
        messages.append({"role": "user", "content": f"Classify the following job description (Original Title: {job['original_job_title']}): {job['job_description']}"})


    try:
        response = client.beta.chat.completions.parse(
            model="gpt-4o",
            messages=messages,
            temperature=0.5, # Lower temperature for more consistent results
            max_tokens=1500, # Increase max tokens to accommodate multiple classifications
            response_format=JobClassificationTable
        )
        # Assuming the response contains a list of classifications
        if response.choices and response.choices[0].message.parsed:
             for item in response.choices[0].message.parsed.job_classification_table:
                 # Extract the original job title from the grouping justification
                 original_title = "Unknown Original Title"
                 # A more robust way would be to have the API return the original title directly,
                 # but given the current Pydantic model, we'll try to extract it from the justification
                 # This approach is still not ideal and should be improved if the API model can be changed.
                 # For now, let's iterate through the original batch to find the matching description
                 # and use its original title.
                 matched_job = next((job for job in batch if job['job_description'] in item.grouping_justification), None)
                 if matched_job:
                     original_title = matched_job['original_job_title']
                 else:
                     # As a fallback, try to find a match based on the new job title or major role group
                     matched_job = next((job for job in batch if item.new_job_title in job['original_job_title'] or item.major_role_group in job['original_job_title']), None)
                     if matched_job:
                         original_title = matched_job['original_job_title']


                 classified_jobs.append({
                     'original_job_title': original_title,
                     'new_job_title': item.new_job_title,
                     'major_role_group': item.major_role_group,
                     'minor_sub_group': item.minor_sub_group,
                     'grouping_justification': item.grouping_justification
                 })

        else:
            errors.append(f"No parsed response for batch starting at index {i}")


    except Exception as e:
        errors.append(f"Error processing batch starting at index {i}: {e}")
        # Implement a delay before the next request in case of rate limits or errors
        time.sleep(60) # Wait for 60 seconds before the next attempt

    # Add a delay between batches to avoid hitting rate limits
    time.sleep(10) # Wait for 10 seconds between batches

# Optionally, convert classified_jobs to a DataFrame
classified_jobs_df = pd.DataFrame(classified_jobs)

# Display the first few classified jobs and any errors encountered
print("Classified Jobs (first 5):")
display(classified_jobs_df.head())
print("\nErrors Encountered:")
print(errors)

## Collect and structure results

### Subtask:
Collect the responses from the API calls and structure them into a pandas DataFrame containing all classified jobs.

**Reasoning**:
I need to collect the API responses and structure them into a pandas DataFrame as requested by the subtask. The `classified_jobs` list already contains the dictionaries with the classified job data from the previous API calls. I just need to convert this list into a DataFrame and display the head and the number of rows to complete the subtask.

In [ ]:
# The classified_jobs list already contains the dictionaries of classified job data from the previous step.
# Convert the list of dictionaries into a pandas DataFrame.
classified_jobs_df = pd.DataFrame(classified_jobs)

# Display the head of the resulting DataFrame to verify its structure and content.
print("Head of the DataFrame with all classified jobs:")
display(classified_jobs_df.head())

# Print the number of rows in the DataFrame to confirm all classified jobs are included.
print(f"\nTotal number of classified jobs: {len(classified_jobs_df)}")

## Save results to csv

### Subtask:
Save the final DataFrame containing all classified jobs to a CSV file.

**Reasoning**:
Save the classified jobs DataFrame to a CSV file without the index.

In [ ]:
classified_jobs_df.to_csv('classified_job_descriptions.csv', index=False)

## Review and analyze results

### Subtask:
Examine the generated classifications in the DataFrame.

**Reasoning**:
Display the first 5 rows of the classified_jobs_df DataFrame, print the number of rows to confirm all 114 original job descriptions have a corresponding classification, and display the value counts for the 'major_role_group' and 'minor_sub_group' columns.

In [ ]:
display(classified_jobs_df.head())
print(f"Number of classified jobs: {len(classified_jobs_df)}")
print("\nMajor Role Group Distribution:")
display(classified_jobs_df['major_role_group'].value_counts())
print("\nMinor Sub-Group Distribution:")
display(classified_jobs_df['minor_sub_group'].value_counts())

**Reasoning**:
Display a random sample of 10 rows from the classified_jobs_df DataFrame to qualitatively assess the classification results and justifications.

In [ ]:
display(classified_jobs_df[['original_job_title', 'new_job_title', 'major_role_group', 'minor_sub_group', 'grouping_justification']].sample(10))

## Summary:

### Data Analysis Key Findings

* The initial dataset "New Sample\_08.07.2025.csv" containing 114 job descriptions was successfully loaded.
* Job descriptions were formatted by concatenating relevant columns for processing.
* The job descriptions were processed in batches using the OpenAI API, and the results were collected.
* The classified results were structured into a pandas DataFrame, which contained columns for original and new job titles, major and minor role groups, and grouping justifications.
* The resulting DataFrame contained {num\_classified\_jobs} classified jobs, which is {comparison\_to\_114} the original 114 records{discrepancy\_note}.
* The major role groups identified included 'Specialist', 'Analyst', and 'Manager' among others.
* The minor sub-groups included various levels (I, II, III) and specific sub-groups.
* The classified results, including new job titles, major/minor groups, and justifications, were saved to a CSV file named "classified\_job\_descriptions.csv".

### Insights or Next Steps

* Investigate why {discrepancy\_reasoning} if all 114 records were not classified and included in the final output.
* Review the classifications for accuracy and potentially refine the prompt or process for the OpenAI API calls to improve results or handle edge cases.

## Copy Results to Google Drive

### Subtask:
Copy the generated results file ('classified_job_descriptions.csv') to a specified folder in Google Drive.

**Reasoning**:
Mount Google Drive to access it from the Colab environment, create the target folder if it doesn't exist, and copy the 'classified_job_descriptions.csv' file to the specified Google Drive folder.

In [ ]:
from google.colab import drive
import os
import shutil
import datetime

# Mount Google Drive
drive.mount('/content/drive')

# Define the base target folder path in Google Drive
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Run Results'

# Generate a unique folder name with a timestamp (UTC)
timestamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
unique_folder_name = f'RUN_{timestamp}'

# Define the full target folder path
target_folder = os.path.join(base_target_folder, unique_folder_name)

# Create the target folder if it doesn't exist
os.makedirs(target_folder, exist_ok=True)

# List of files to copy
# This includes the input data, resource files, and the final output file
files_to_copy = [
    '/content/New Sample_08.07.2025.csv',
    '/content/MNPS Roles.csv',
    '/content/Competency Extended Descriptions.csv',
    '/content/MNPS KSACs.csv',
    '/content/Korn_Ferry Lominger 38 Competencies.csv',
    'classified_job_descriptions.csv' # This file is in the current directory
]

# Copy each file to Google Drive
for file_path in files_to_copy:
    try:
        # Get the base name of the file
        file_name = os.path.basename(file_path)
        destination_path = os.path.join(target_folder, file_name)
        shutil.copy(file_path, destination_path)
        print(f"Successfully copied {file_name} to {destination_path}")
    except FileNotFoundError:
        print(f"Error: {file_name} not found at {file_path}.")
    except Exception as e:
        print(f"Error copying file {file_name}: {e}")